# Phase 6: Dashboard Data Exports

> **CRISP-DM Phase**: 6 of 6 — Deployment  
> **Purpose**: Export all CSV files needed for the Tableau State Deep Dive Dashboard

## Files to Export

1. `fact_opioid_analysis.csv` — Main fact table (already exported in Phase 3)
2. `state_drug_breakdown.csv` — Top drugs by claims for each state
3. `state_specialty_breakdown.csv` — Top specialties by claims for each state
4. `national_benchmarks.csv` — National averages for the radar chart comparison
5. `state_comparison_metrics.csv` — Each state's percentile rank for gauge charts

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sqlalchemy import create_engine, text

PROJECT_ROOT = Path('.').resolve().parent
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_DIR.mkdir(exist_ok=True)

engine = create_engine('postgresql://yashcomputers@localhost:5432/opioid_analysis')
print(f'Connected. Exports will go to: {OUTPUT_DIR}')

Connected. Exports will go to: /Users/krunal/opioid-pipeline-analysis/data/processed


---
## Export 1: State Drug Breakdown

Top 10 drugs by claims for EACH state. This powers the "Top 5 Drugs in This State" chart.

In [2]:
# ============================================================
# Export 1: Top 10 drugs per state
# ============================================================

state_drugs = pd.read_sql("""
    WITH ranked_drugs AS (
        SELECT
            d.state_abbrev,
            d.state_name,
            c.gnrc_name,
            SUM(c.tot_clms) AS total_claims,
            SUM(c.tot_drug_cst) AS total_cost,
            COUNT(DISTINCT c.prscrbr_npi) AS prescriber_count,
            ROW_NUMBER() OVER (
                PARTITION BY d.state_abbrev 
                ORDER BY SUM(c.tot_clms) DESC
            ) AS drug_rank
        FROM stg_cms_partd c
        JOIN dim_state d ON TRIM(c.prscrbr_state_abrvtn) = d.state_abbrev
        WHERE c.gnrc_name NOT IN (
            'Tiotropium Bromide', 'Ipratropium Bromide',
            'Ipratropium/Albuterol Sulfate', 'Tiotropium Br/Olodaterol Hcl',
            'Apomorphine Hcl'
        )
        GROUP BY d.state_abbrev, d.state_name, c.gnrc_name
    )
    SELECT state_abbrev, state_name, gnrc_name, 
           total_claims, total_cost, prescriber_count, drug_rank
    FROM ranked_drugs
    WHERE drug_rank <= 10
    ORDER BY state_abbrev, drug_rank;
""", engine)

# Calculate percentage of state total for each drug
state_totals = state_drugs.groupby('state_abbrev')['total_claims'].transform('sum')
state_drugs['pct_of_state_claims'] = (state_drugs['total_claims'] / state_totals * 100).round(2)

state_drugs.to_csv(OUTPUT_DIR / 'state_drug_breakdown.csv', index=False)

print(f'State Drug Breakdown: {len(state_drugs)} rows ({state_drugs["state_abbrev"].nunique()} states × up to 10 drugs)')
print(f'\nSample — West Virginia top 5:')
display(state_drugs[state_drugs['state_abbrev'].str.strip() == 'WV'].head())

State Drug Breakdown: 510 rows (51 states × up to 10 drugs)

Sample — West Virginia top 5:


,state_abbrev,state_name,gnrc_name,total_claims,total_cost,prescriber_count,drug_rank,pct_of_state_claims
490,WV,West Virginia,Hydrocodone/Acetaminophen,172623,3316279.53,1570,1,37.23
491,WV,West Virginia,Tramadol Hcl,111895,848636.28,1498,2,24.13
492,WV,West Virginia,Oxycodone Hcl/Acetaminophen,57224,2044750.07,771,3,12.34
493,WV,West Virginia,Buprenorphine Hcl/Naloxone Hcl,39655,4963205.69,208,4,8.55
494,WV,West Virginia,Oxycodone Hcl,31127,2558766.41,508,5,6.71


---
## Export 2: State Specialty Breakdown

Top 10 specialties by claims for EACH state. This powers the "Top 5 Specialties in This State" chart.

In [3]:
# ============================================================
# Export 2: Top 10 specialties per state
# ============================================================

state_specialties = pd.read_sql("""
    WITH ranked_specialties AS (
        SELECT
            d.state_abbrev,
            d.state_name,
            -- Consolidate Family Practice / Family Medicine
            CASE 
                WHEN c.prscrbr_type = 'Family Medicine' THEN 'Family Practice'
                ELSE c.prscrbr_type
            END AS specialty,
            SUM(c.tot_clms) AS total_claims,
            COUNT(DISTINCT c.prscrbr_npi) AS prescriber_count,
            ROW_NUMBER() OVER (
                PARTITION BY d.state_abbrev
                ORDER BY SUM(c.tot_clms) DESC
            ) AS specialty_rank
        FROM stg_cms_partd c
        JOIN dim_state d ON TRIM(c.prscrbr_state_abrvtn) = d.state_abbrev
        WHERE c.gnrc_name NOT IN (
            'Tiotropium Bromide', 'Ipratropium Bromide',
            'Ipratropium/Albuterol Sulfate', 'Tiotropium Br/Olodaterol Hcl',
            'Apomorphine Hcl'
        )
        GROUP BY d.state_abbrev, d.state_name,
                 CASE WHEN c.prscrbr_type = 'Family Medicine' THEN 'Family Practice'
                      ELSE c.prscrbr_type END
    )
    SELECT state_abbrev, state_name, specialty,
           total_claims, prescriber_count, specialty_rank
    FROM ranked_specialties
    WHERE specialty_rank <= 10
    ORDER BY state_abbrev, specialty_rank;
""", engine)

# Add primary care vs specialist flag
primary_care = ['Family Practice', 'Internal Medicine', 'Nurse Practitioner',
                'Physician Assistant', 'General Practice']
state_specialties['provider_category'] = state_specialties['specialty'].apply(
    lambda x: 'Primary Care' if x in primary_care else 'Specialist'
)

# Calculate claims per prescriber (intensity)
state_specialties['claims_per_prescriber'] = (
    state_specialties['total_claims'] / state_specialties['prescriber_count']
).round(1)

state_specialties.to_csv(OUTPUT_DIR / 'state_specialty_breakdown.csv', index=False)

print(f'State Specialty Breakdown: {len(state_specialties)} rows')
print(f'\nSample — Kentucky top 5:')
display(state_specialties[state_specialties['state_abbrev'].str.strip() == 'KY'].head())

State Specialty Breakdown: 510 rows

Sample — Kentucky top 5:


,state_abbrev,state_name,specialty,total_claims,prescriber_count,specialty_rank,provider_category,claims_per_prescriber
170,KY,Kentucky,Family Practice,364753,1013,1,Primary Care,360.1
171,KY,Kentucky,Nurse Practitioner,267343,1539,2,Primary Care,173.7
172,KY,Kentucky,Internal Medicine,234143,690,3,Primary Care,339.3
173,KY,Kentucky,Pain Management,152469,42,4,Specialist,3630.2
174,KY,Kentucky,Anesthesiology,110667,56,5,Specialist,1976.2


---
## Export 3: National Benchmarks

National averages and percentiles for the radar chart and gauge comparisons.

In [4]:
# ============================================================
# Export 3: National benchmarks
# ============================================================

fact_df = pd.read_sql('SELECT * FROM fact_opioid_analysis', engine)
analysis_df = fact_df[fact_df['death_rate_per_100k'].notna()].copy()

# Metrics for radar chart and gauges
metrics = ['opioid_rx_rate', 'death_rate_per_100k', 'w_pct_poverty',
           'w_pct_unemployed', 'w_pct_uninsured', 'w_gini_index']

benchmarks = pd.DataFrame({
    'metric': metrics,
    'national_mean': [analysis_df[m].mean() for m in metrics],
    'national_median': [analysis_df[m].median() for m in metrics],
    'national_min': [analysis_df[m].min() for m in metrics],
    'national_max': [analysis_df[m].max() for m in metrics],
    'national_q25': [analysis_df[m].quantile(0.25) for m in metrics],
    'national_q75': [analysis_df[m].quantile(0.75) for m in metrics],
}).round(4)

# Add friendly labels
label_map = {
    'opioid_rx_rate': 'Prescribing Rate (per 1K)',
    'death_rate_per_100k': 'Death Rate (per 100K)',
    'w_pct_poverty': 'Poverty Rate (%)',
    'w_pct_unemployed': 'Unemployment Rate (%)',
    'w_pct_uninsured': 'Uninsured Rate (%)',
    'w_gini_index': 'GINI Index',
}
benchmarks['label'] = benchmarks['metric'].map(label_map)

benchmarks.to_csv(OUTPUT_DIR / 'national_benchmarks.csv', index=False)

print('National Benchmarks:')
display(benchmarks)

National Benchmarks:


,metric,national_mean,national_median,national_min,national_max,national_q25,national_q75,label
0,opioid_rx_rate,181.9939,169.5400,78.4400,337.31,144.3700,221.9800,Prescribing Rate (per 1K)
1,death_rate_per_100k,5.8917,5.2800,1.2500,17.21,3.8500,8.4100,Death Rate (per 100K)
2,w_pct_poverty,12.5032,12.3900,7.4500,19.66,10.3200,14.1400,Poverty Rate (%)
3,w_pct_unemployed,5.2568,5.3400,3.5800,7.43,4.5500,5.8100,Unemployment Rate (%)
4,w_pct_uninsured,7.9720,7.6200,2.7300,17.30,5.6200,9.7500,Uninsured Rate (%)
5,w_gini_index,0.4531,0.4545,0.4168,0.52,0.4376,0.4642,GINI Index


---
## Export 4: State Comparison Metrics

Each state's percentile rank across all metrics — powers the gauge/bullet charts showing where a state falls relative to the national distribution.

In [5]:
# ============================================================
# Export 4: State percentile ranks for gauge charts
# ============================================================

comparison = fact_df[['state_abbrev', 'state_name', 'census_region',
                      'opioid_rx_rate', 'death_rate_per_100k',
                      'w_pct_poverty', 'w_pct_unemployed',
                      'w_pct_uninsured', 'w_gini_index',
                      'state_population', 'combined_risk_score',
                      'combined_risk_rank', 'prescribing_quartile',
                      'sdoh_risk_score']].copy()

# Calculate percentile rank for each metric (0-100, higher = worse)
for col in metrics:
    valid = comparison[col].notna()
    comparison.loc[valid, f'{col}_percentile'] = (
        comparison.loc[valid, col].rank(pct=True) * 100
    ).round(1)

comparison.to_csv(OUTPUT_DIR / 'state_comparison_metrics.csv', index=False)

print(f'State Comparison Metrics: {len(comparison)} states × {len(comparison.columns)} columns')
print(f'\nSample — Top 5 risk states:')
display(comparison.sort_values('combined_risk_rank').head()[[
    'state_abbrev', 'state_name', 'opioid_rx_rate', 'death_rate_per_100k',
    'sdoh_risk_score', 'combined_risk_rank',
    'opioid_rx_rate_percentile', 'death_rate_per_100k_percentile'
]])

State Comparison Metrics: 51 states × 20 columns

Sample — Top 5 risk states:


,state_abbrev,state_name,opioid_rx_rate,death_rate_per_100k,sdoh_risk_score,combined_risk_rank,opioid_rx_rate_percentile,death_rate_per_100k_percentile
5,WV,West Virginia,261.11,17.21,61.85,1.0,90.2,100.0
1,KY,Kentucky,337.31,10.53,50.24,2.0,98.0,97.6
2,TN,Tennessee,309.69,9.11,53.35,3.0,96.1,87.8
12,SC,South Carolina,228.25,8.90,56.51,4.0,76.5,85.4
32,NM,New Mexico,156.03,10.01,73.49,5.0,37.3,95.1


---
## Export 5: Verify the main fact table is current

In [6]:
# ============================================================
# Verify fact_opioid_analysis.csv exists and is current
# ============================================================

fact_path = OUTPUT_DIR / 'fact_opioid_analysis.csv'
if fact_path.exists():
    fact_check = pd.read_csv(fact_path)
    print(f'✅ fact_opioid_analysis.csv exists: {len(fact_check)} rows × {len(fact_check.columns)} columns')
else:
    print('⚠️ fact_opioid_analysis.csv not found — exporting now...')
    fact_df.to_csv(fact_path, index=False)
    print(f'✅ Exported: {len(fact_df)} rows')

✅ fact_opioid_analysis.csv exists: 51 rows × 31 columns


---
## Summary of All Exports

In [7]:
# ============================================================
# Final inventory
# ============================================================
import os

print('DASHBOARD DATA FILES')
print('=' * 65)
print(f'Location: {OUTPUT_DIR}')
print()

for f in sorted(OUTPUT_DIR.glob('*.csv')):
    size = os.path.getsize(f) / 1024
    rows = len(pd.read_csv(f))
    print(f'  {f.name:40s} {size:>8.1f} KB  ({rows:,} rows)')

print(f'\nAll files ready for Tableau import.')

DASHBOARD DATA FILES
Location: /Users/krunal/opioid-pipeline-analysis/data/processed

  fact_opioid_analysis.csv                     12.3 KB  (51 rows)
  feature_importance.csv                        0.2 KB  (6 rows)
  national_benchmarks.csv                       0.5 KB  (6 rows)
  state_comparison_metrics.csv                  5.7 KB  (51 rows)
  state_drug_breakdown.csv                     30.3 KB  (510 rows)
  state_specialty_breakdown.csv                31.0 KB  (510 rows)

All files ready for Tableau import.
